### Facebook AI Similarity Search (FAISS)

In [1]:
from langchain_community.document_loaders import TextLoader
from langchain_community.embeddings import OllamaEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import CharacterTextSplitter

c:\Users\viren\anaconda3\envs\langchain_venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
loader = TextLoader('speech.txt')
text_documents = loader.load()
text_documents

[Document(metadata={'source': 'speech.txt'}, page_content='The world must be made safe for democracy. Its peace must be planted upon the tested foundations of political liberty. We have no selfish ends to serve. We desire no conquest, no dominion. We seek no indemnities for ourselves, no material compensation for the sacrifices we shall freely make. We are but one of the champions of the rights of mankind. We shall be satisfied when those rights have been made as secure as the faith and the freedom of nations can make them.\n\nJust because we fight without rancor and without selfish object, seeking nothing for ourselves but what we shall wish to share with all free peoples, we shall, I feel confident, conduct our operations as belligerents without passion and ourselves observe with proud punctilio the principles of right and of fair play we profess to be fighting for.\n\n…\n\nIt will be all the easier for us to conduct ourselves as belligerents in a high spirit of right and fairness be

In [3]:
text_splitter = CharacterTextSplitter(separator="\n\n", chunk_size = 1000, chunk_overlap = 30)
splitted_text_documents = text_splitter.split_documents(text_documents)
splitted_text_documents

[Document(metadata={'source': 'speech.txt'}, page_content='The world must be made safe for democracy. Its peace must be planted upon the tested foundations of political liberty. We have no selfish ends to serve. We desire no conquest, no dominion. We seek no indemnities for ourselves, no material compensation for the sacrifices we shall freely make. We are but one of the champions of the rights of mankind. We shall be satisfied when those rights have been made as secure as the faith and the freedom of nations can make them.\n\nJust because we fight without rancor and without selfish object, seeking nothing for ourselves but what we shall wish to share with all free peoples, we shall, I feel confident, conduct our operations as belligerents without passion and ourselves observe with proud punctilio the principles of right and of fair play we profess to be fighting for.\n\n…'),
 Document(metadata={'source': 'speech.txt'}, page_content='…\n\nIt will be all the easier for us to conduct our

In [4]:
embeddings = OllamaEmbeddings(model='qwen3-embedding:4b')
embeddings

C:\Users\viren\AppData\Local\Temp\ipykernel_11928\1123023006.py:1: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaEmbeddings``.
  embeddings = OllamaEmbeddings(model='qwen3-embedding:4b')


OllamaEmbeddings(base_url='http://localhost:11434', model='qwen3-embedding:4b', embed_instruction='passage: ', query_instruction='query: ', mirostat=None, mirostat_eta=None, mirostat_tau=None, num_ctx=None, num_gpu=None, num_thread=None, repeat_last_n=None, repeat_penalty=None, temperature=None, stop=None, tfs_z=None, top_k=None, top_p=None, show_progress=False, headers=None, model_kwargs=None)

In [5]:
# Embedding and Storing the Document in the VectorStoreDB 
db = FAISS.from_documents(splitted_text_documents, embeddings)
db

In [7]:
# Querying the VectorStoreDB
query_text = "What does the speaker believe is the main reason the United States should enter the war?"
docs = db.similarity_search(query_text)
docs

[Document(id='bc18dcfa-3e91-44ef-89f0-555872706190', metadata={'source': 'speech.txt'}, page_content='It is a distressing and oppressive duty, gentlemen of the Congress, which I have performed in thus addressing you. There are, it may be, many months of fiery trial and sacrifice ahead of us. It is a fearful thing to lead this great peaceful people into war, into the most terrible and disastrous of all wars, civilization itself seeming to be in the balance. But the right is more precious than peace, and we shall fight for the things which we have always carried nearest our hearts—for democracy, for the right of those who submit to authority to have a voice in their own governments, for the rights and liberties of small nations, for a universal dominion of right by such a concert of free peoples as shall bring peace and safety to all nations and make the world itself at last free.'),
 Document(id='1c0220fd-f14e-4acb-afda-71a4f6c0b646', metadata={'source': 'speech.txt'}, page_content='To 

In [8]:
docs[0].page_content

'It is a distressing and oppressive duty, gentlemen of the Congress, which I have performed in thus addressing you. There are, it may be, many months of fiery trial and sacrifice ahead of us. It is a fearful thing to lead this great peaceful people into war, into the most terrible and disastrous of all wars, civilization itself seeming to be in the balance. But the right is more precious than peace, and we shall fight for the things which we have always carried nearest our hearts—for democracy, for the right of those who submit to authority to have a voice in their own governments, for the rights and liberties of small nations, for a universal dominion of right by such a concert of free peoples as shall bring peace and safety to all nations and make the world itself at last free.'

### Converting VectorStoreDB as a Retriever

In [10]:
retriever = db.as_retriever()
docs = retriever.invoke(query_text)
docs[0].page_content

'It is a distressing and oppressive duty, gentlemen of the Congress, which I have performed in thus addressing you. There are, it may be, many months of fiery trial and sacrifice ahead of us. It is a fearful thing to lead this great peaceful people into war, into the most terrible and disastrous of all wars, civilization itself seeming to be in the balance. But the right is more precious than peace, and we shall fight for the things which we have always carried nearest our hearts—for democracy, for the right of those who submit to authority to have a voice in their own governments, for the rights and liberties of small nations, for a universal dominion of right by such a concert of free peoples as shall bring peace and safety to all nations and make the world itself at last free.'

#### Similarity Search With Score (in FAISS VectorStoreDB )

In [11]:
docs_and_score = db.similarity_search_with_score(query_text)
docs_and_score

[(Document(id='bc18dcfa-3e91-44ef-89f0-555872706190', metadata={'source': 'speech.txt'}, page_content='It is a distressing and oppressive duty, gentlemen of the Congress, which I have performed in thus addressing you. There are, it may be, many months of fiery trial and sacrifice ahead of us. It is a fearful thing to lead this great peaceful people into war, into the most terrible and disastrous of all wars, civilization itself seeming to be in the balance. But the right is more precious than peace, and we shall fight for the things which we have always carried nearest our hearts—for democracy, for the right of those who submit to authority to have a voice in their own governments, for the rights and liberties of small nations, for a universal dominion of right by such a concert of free peoples as shall bring peace and safety to all nations and make the world itself at last free.'),
  np.float32(0.71902454)),
 (Document(id='1c0220fd-f14e-4acb-afda-71a4f6c0b646', metadata={'source': 'sp

#### Querying using Vectors

In [12]:
embedding_vectors = embeddings.embed_query(query_text)
embedding_vectors

[-0.00028478240710683167,
 0.048127755522727966,
 0.042158592492341995,
 -0.002847850788384676,
 -0.0014286828227341175,
 0.026858938857913017,
 0.024243367835879326,
 -0.017819419503211975,
 0.01883981190621853,
 -0.030270447954535484,
 0.0006537343724630773,
 -0.008172811008989811,
 0.00601669168099761,
 -0.004293926525861025,
 0.06160635128617287,
 0.024545570835471153,
 -0.022257357835769653,
 0.041230909526348114,
 -0.048190802335739136,
 0.0038338215090334415,
 -0.01768566481769085,
 0.04362333565950394,
 0.05528198927640915,
 0.0016252163331955671,
 0.07435091584920883,
 0.01763025112450123,
 -0.02498587593436241,
 -0.06697133928537369,
 0.04037507623434067,
 0.025269879028201103,
 -0.0078025260008871555,
 -0.01173956599086523,
 -0.009604748338460922,
 0.010304031893610954,
 0.002747973660007119,
 -0.010500574484467506,
 0.004109556321054697,
 0.00030064809834584594,
 0.01315539050847292,
 0.013412762433290482,
 0.005114512052386999,
 -0.06443822383880615,
 0.027938295155763626,

In [13]:
docs = db.similarity_search_by_vector(embedding_vectors)
docs

[Document(id='bc18dcfa-3e91-44ef-89f0-555872706190', metadata={'source': 'speech.txt'}, page_content='It is a distressing and oppressive duty, gentlemen of the Congress, which I have performed in thus addressing you. There are, it may be, many months of fiery trial and sacrifice ahead of us. It is a fearful thing to lead this great peaceful people into war, into the most terrible and disastrous of all wars, civilization itself seeming to be in the balance. But the right is more precious than peace, and we shall fight for the things which we have always carried nearest our hearts—for democracy, for the right of those who submit to authority to have a voice in their own governments, for the rights and liberties of small nations, for a universal dominion of right by such a concert of free peoples as shall bring peace and safety to all nations and make the world itself at last free.'),
 Document(id='1c0220fd-f14e-4acb-afda-71a4f6c0b646', metadata={'source': 'speech.txt'}, page_content='To 

#### Saving and Loading VectorStoreDB locally

In [14]:
db.save_local("faiss_index")

In [15]:
new_db = FAISS.load_local("faiss_index", embeddings, allow_dangerous_deserialization=True)
new_db

In [16]:
docs = new_db.similarity_search(query_text)
docs[0].page_content

'It is a distressing and oppressive duty, gentlemen of the Congress, which I have performed in thus addressing you. There are, it may be, many months of fiery trial and sacrifice ahead of us. It is a fearful thing to lead this great peaceful people into war, into the most terrible and disastrous of all wars, civilization itself seeming to be in the balance. But the right is more precious than peace, and we shall fight for the things which we have always carried nearest our hearts—for democracy, for the right of those who submit to authority to have a voice in their own governments, for the rights and liberties of small nations, for a universal dominion of right by such a concert of free peoples as shall bring peace and safety to all nations and make the world itself at last free.'